## 📀 Gold – Movies View

### Goal
Provide a clean final dataset for analytics using Netflix as the base and IMDB as enrichment.

---

### Join principle
A LEFT JOIN is used so every Netflix movie is preserved.  
If enrichment is not available, the fields remain NULL.

---

### Director
Only one director is needed, so the first value from the IMDB array is selected, as person_id is not provided in the dataset to bring in only least persion_id available.

---

### Ratings & votes
IMDB values are preferred.  
If they are missing, Netflix values are used instead.

---

### Why build a VIEW directly on silver?
For this assessment, creating a view is more practical than materialising gold tables.

Reasons:
- keeps the solution simple  
- avoids extra storage  
- no additional pipeline orchestration  
- logic stays transparent and easy to review  
- results are always up to date with silver  

If needed in a real production system, this logic could later be materialised into tables for performance.

---

### Result
One row per Netflix movie with best available enrichment, without losing data.

In [0]:
Jashu@4321

In [0]:
%sql
CREATE OR REPLACE VIEW gold_catalog.media_analytics.v_movies AS
WITH
/* ------------------------------------------------------------
   1) Netflix is my base dataset.
   I must return ALL Netflix movies.
   IMDB is only enrichment.
   ------------------------------------------------------------ */
netflix_src AS (
  SELECT
    id,
    title,
    type,
    description,
    CAST(releaseYear AS INT) AS releaseYear,
    ageCertification,
    runTime,
    runTimeMinutes,
    genresArr,
    productionCountriesArr,
    seasons,
    imdbId,
    imdbScore AS netflixImdbScore,
    imdbVotes AS netflixImdbVotes,
    tmdbPopularity,
    tmdbScore,
    rescuedData,
    sourceFilePath,
    ingestTs,

    /* 
       I normalise titles so joins are more tolerant.
       Remove punctuation, make lowercase, compress spaces.
    */
    regexp_replace(
      regexp_replace(lower(trim(title)), '[^a-z0-9 ]', ''),
      '\\s+',
      ' '
    ) AS titleKeyNorm
  FROM silver_catalog.silver.silverNetflixTitles
),

/* ------------------------------------------------------------
   2) IMDB data used for enrichment (rating, votes, directors)
   ------------------------------------------------------------ */
imdb_src AS (
  SELECT
    title,
    try_cast(releaseYear AS INT) AS releaseYear,
    imdbScore AS imdbImdbScore,
    imdbVotes AS imdbImdbVotes,
    directorsArr,

    /*
      IMDB titles sometimes came with prefixes like:
      "10. Movie Name"
      so I remove leading numbers before building the key.
    */
    regexp_replace(
      regexp_replace(
        lower(trim(regexp_replace(title, '^\\s*\\d+\\.\\s*', ''))),
        '[^a-z0-9 ]',
        ''
      ),
      '\\s+',
      ' '
    ) AS titleKeyNorm
  FROM silver_catalog.silver.silverImdbMergedMoviesData
),

/* ------------------------------------------------------------
   3) LEFT JOIN so I never lose Netflix titles.
   If IMDB doesn't match, enrichment fields stay NULL.
   ------------------------------------------------------------ */
candidates AS (
  SELECT
    n.*,
    i.imdbImdbScore,
    i.imdbImdbVotes,

    /* 
       Requirement only needs one director.
       Keeping it simple → take the first element.
    */
    element_at(i.directorsArr, 1) AS directorFromImdb,

    /*
      Prefer IMDB rating & votes.
      If missing, fallback to Netflix values.
    */
    COALESCE(i.imdbImdbScore, n.netflixImdbScore) AS chosenImdbScore,
    CAST(COALESCE(i.imdbImdbVotes, n.netflixImdbVotes) AS INT) AS chosenImdbVotes

  FROM netflix_src n
  LEFT JOIN imdb_src i
    ON n.titleKeyNorm = i.titleKeyNorm
),

/* ------------------------------------------------------------
   4) If multiple IMDB matches happen,
   I keep the highest rated one.
   ------------------------------------------------------------ */
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY id
      ORDER BY chosenImdbScore DESC NULLS LAST
    ) AS rn
  FROM candidates
)

/* ------------------------------------------------------------
   5) Final gold output
   ------------------------------------------------------------ */
SELECT
  id,
  title,
  type,
  description,
  releaseYear,
  runTimeMinutes,
  ageCertification,

  directorFromImdb AS director,

  chosenImdbScore AS imdbScore,
  chosenImdbVotes AS imdbVotes,

  -- keeping arrays, useful for analytics
  runTime,
  genresArr,
  productionCountriesArr,
  seasons,
  imdbId,
  tmdbPopularity,
  tmdbScore,
  rescuedData,

  -- lineage
  sourceFilePath,
  ingestTs

FROM ranked
WHERE rn = 1;

In [0]:
%sql
SELECT
*
FROM gold_catalog.media_analytics.v_movies;

In [0]:
%sql
select * from gold_catalog.media_analytics.v_movies

In [0]:
%sql
SELECT title, releaseYear
FROM silver_catalog.silver.silverImdbMergedMoviesData
WHERE lower(title) LIKE '%madagascar 3%'
LIMIT 50;

In [0]:
%sql
SELECT title, releaseYear
FROM silver_catalog.silver.silverImdbMergedMoviesData
WHERE lower(title) LIKE '%madea%'
  AND lower(title) LIKE '%witness%'
LIMIT 50;

In [0]:
%sql
select title, releaseYear
from silver_catalog.silver.silverNetflixTitles
limit 5;

In [0]:
%sql


-- then copy one title from above and run:
select title, releaseYear
from silver_catalog.silver.silverImdbMergedMoviesData
where title like '%Five Came Back: The Reference Films%'
limit 20;